# SCAGE Cliff Attention 視覺化

此 notebook 僅用於 cliff finetune 模型 attention 抽取與視覺化：

1. atomic attention score heatmap（支援逐層）
2. 2D 分子圖（紅色強度 + 0-1 colorbar）

> 請在 Docker `scage` container（路徑 `/workspace/SCAGE-master`）內執行。


In [1]:
# --- Notebook 啟動位置修正 ---
# 目的：避免在 Notebook/ 目錄執行時，無法 import 專案模組
import os
import sys
from pathlib import Path


def _resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for c in candidates:
        if (c / '_config.py').exists() and (c / 'models').exists() and (c / 'data_process').exists():
            return c.resolve()
    raise RuntimeError('Cannot locate project root. Please open notebook under SCAGE project.')


PROJECT_ROOT = _resolve_project_root()
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'[Init] PROJECT_ROOT = {PROJECT_ROOT}')
print(f'[Init] CWD switched to = {Path.cwd()}')

[Init] PROJECT_ROOT = /workspace
[Init] CWD switched to = /workspace


In [2]:
import os
from pathlib import Path
from copy import deepcopy
from typing import Dict, List

import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D

from _config import pdir, get_downstream_task_names
from data_process.compound_tools import CompoundKit, mol_to_data_pkl
from data_process.data_collator import collator_finetune_pkl
from data_process.function_group_constant import nfg
from models.scage import Scage
from models.layers.encoder import MultiScaleAttention
from utils.global_var_util import GlobalVar
from utils.userconfig_util import config_current_user, config_dataset_form

plt.rcParams['figure.dpi'] = 140
torch.set_printoptions(sci_mode=False, precision=4)
np.set_printoptions(precision=4, suppress=True)

print('cwd:', os.getcwd())

cwd: /workspace


In [ ]:
def plot_raw_multiscale_attn_list(attn_per_layer: List[List[torch.Tensor]], title_prefix: str = 'Raw MultiScaleAttention attn_list', batch_idx: int = 0, max_cols: int = 4):
    """Plot full token-token raw attention heatmaps after integrating heads/scales per layer and all layers."""
    if len(attn_per_layer) == 0:
        raise RuntimeError('No attention captured. Check hook registration.')

    layer_mats = []
    for layer_idx, layer_data in enumerate(attn_per_layer):
        scale_mats = []
        raw_shapes = []
        for scale_idx, attn in enumerate(layer_data):
            raw = attn.detach().cpu()
            if raw.ndim != 4:
                raise ValueError(f'Expected attn tensor [batch, heads, tokens, tokens], got shape={tuple(raw.shape)}')
            if batch_idx < 0 or batch_idx >= raw.shape[0]:
                raise ValueError(f'batch_idx out of range: {batch_idx}, batch_size={raw.shape[0]}')

            # Integrate heads only by averaging; keep the original attention values otherwise.
            scale_mats.append(raw[batch_idx].mean(dim=0).numpy())
            raw_shapes.append(tuple(raw.shape))

        layer_mat = np.mean(scale_mats, axis=0)  # integrate multiscale branches into one matrix per layer
        layer_mats.append(layer_mat)
        print(
            f'raw attn layer={layer_idx}, shapes={raw_shapes}, '
            f'head/scale-integrated min={layer_mat.min():.6g}, max={layer_mat.max():.6g}'
        )

    integrated_mat = np.mean(layer_mats, axis=0)
    plot_mats = layer_mats + [integrated_mat]
    plot_titles = [f'Layer {idx}' for idx in range(len(layer_mats))] + ['All layers integrated']

    n_plots = len(plot_mats)
    n_cols = min(max_cols, n_plots)
    n_rows = int(np.ceil(n_plots / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.2 * n_cols, 3.6 * n_rows), squeeze=False)

    for plot_idx, ax in enumerate(axes.flat):
        if plot_idx >= n_plots:
            ax.axis('off')
            continue
        mat = plot_mats[plot_idx]
        sns.heatmap(
            mat,
            cmap='Blues',
            square=True,
            ax=ax,
            cbar=True,
            xticklabels=True,
            yticklabels=True,
        )
        ax.set_title(f'{plot_titles[plot_idx]} | min={mat.min():.4g}, max={mat.max():.4g}')
        ax.set_xlabel('Key token index')
        ax.set_ylabel('Query token index')

    fig.suptitle(f'{title_prefix} | raw head/scale-integrated full attention matrices', y=1.02)
    fig.tight_layout()
    plt.show()
    print(
        f'raw attn all layers integrated, shape={integrated_mat.shape}, '
        f'min={integrated_mat.min():.6g}, max={integrated_mat.max():.6g}'
    )
    return [fig]

In [3]:
def build_single_batch(smiles: str, device: str = 'cuda') -> Dict[str, torch.Tensor]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f'Invalid SMILES: {smiles}')
    data = mol_to_data_pkl(mol)
    data['smiles'] = smiles
    data['label'] = np.array([0.0], dtype=np.float32)  # collator 需要 label
    batch = collator_finetune_pkl([data])
    batch = {k: v.to(device) for k, v in batch.items() if v is not None and not isinstance(v, list)}
    batch['edge_weight'] = None
    return batch
def register_attention_hooks(model: torch.nn.Module, sink: List[List[torch.Tensor]]):
    hooks = []
    def _hook(_, __, out):
        # MultiScaleAttention.forward -> (x, attn_list)
        attn_list = out[1]
        sink.append([a.detach().cpu() for a in attn_list])
    for m in model.modules():
        if isinstance(m, MultiScaleAttention):
            hooks.append(m.register_forward_hook(_hook))
    return hooks

def _minmax_normalize_scores(score: np.ndarray, axis=None) -> np.ndarray:
    score = np.asarray(score, dtype=float)
    score_min = score.min(axis=axis, keepdims=True)
    score_max = score.max(axis=axis, keepdims=True)
    return (score - score_min) / (score_max - score_min + 1e-12)


def aggregate_attention(attn_per_layer: List[List[torch.Tensor]], n_atoms: int, layer_idx: int = None):
    if len(attn_per_layer) == 0:
        raise RuntimeError('No attention captured. Check hook registration.')
    # 每層內先對 dist_bar 與 heads 平均
    layer_token_mats = []
    for layer_data in attn_per_layer:
        dist_branch_mats = []
        for dist_attn in layer_data:
            # shape: [batch, heads, tokens, tokens], 單樣本取 batch=0
            mat = dist_attn[0].mean(dim=0).numpy()
            dist_branch_mats.append(mat)
        layer_token_mats.append(np.mean(dist_branch_mats, axis=0))
    # layer_idx=None 代表所有層平均；否則取指定層
    if layer_idx is None:
        full = np.mean(layer_token_mats, axis=0)
    else:
        if layer_idx < 0 or layer_idx >= len(layer_token_mats):
            raise ValueError(f'layer_idx out of range: {layer_idx}, total layers={len(layer_token_mats)}')
        full = layer_token_mats[layer_idx]
    # token 0 是 super node；真實 atoms 從 1 開始
    raw_atom_atom = full[1:n_atoms+1, 1:n_atoms+1]
    atom_atom = _minmax_normalize_scores(raw_atom_atom)
    # 回傳每一層的 atom-atom attention，方便逐層視覺化
    layer_atom_mats = []
    for mat in layer_token_mats:
        aa = mat[1:n_atoms+1, 1:n_atoms+1]
        layer_atom_mats.append(_minmax_normalize_scores(aa))
    raw_atom_importance = raw_atom_atom.mean(axis=0)
    atom_importance = _minmax_normalize_scores(raw_atom_importance, axis=-1)
    return atom_atom, raw_atom_importance, atom_importance, layer_atom_mats


In [ ]:
def draw_attention_on_molecule(smiles: str, atom_scores: np.ndarray, size=(700, 420), contrast_gamma: float = None):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f'Invalid SMILES: {smiles}')
    Chem.rdDepictor.Compute2DCoords(mol)
    n_atoms = mol.GetNumAtoms()
    if len(atom_scores) != n_atoms:
        raise ValueError(f'atom_scores length {len(atom_scores)} != num_atoms {n_atoms}')
    if contrast_gamma is None:
        contrast_gamma = globals().get('CLIFF_CONTRAST_GAMMA', 0.4)
    # atom_scores 已由呼叫端用 min-max normalization 處理到 0~1。
    atom_scores = np.clip(np.asarray(atom_scores, dtype=float), 0.0, 1.0)
    # 對比分布增強：gamma < 1 會拉開高分區間，讓高低差異更明顯
    # 若想更強烈可用 0.4；較溫和可用 0.7
    atom_scores = np.power(atom_scores, contrast_gamma)
    # Atom 顏色：紅色 alpha = atom score
    highlight_atoms = list(range(n_atoms))
    atom_cols = {
        i: (1.0, 0.0, 0.0, float(atom_scores[i]))
        for i in highlight_atoms
    }
    # Bond 顏色：edge score = 兩端 atom score 平均
    highlight_bonds = []
    bond_cols = {}
    for bond in mol.GetBonds():
        b_idx = bond.GetIdx()
        a1 = bond.GetBeginAtomIdx()
        a2 = bond.GetEndAtomIdx()
        edge_score = float((atom_scores[a1] + atom_scores[a2]) / 2.0)
        highlight_bonds.append(b_idx)
        bond_cols[b_idx] = (1.0, 0.0, 0.0, edge_score)
    drawer = rdMolDraw2D.MolDraw2DCairo(size[0], size[1])
    opts = drawer.drawOptions()
    opts.useBWAtomPalette()  # 原子元素顏色統一黑色（如 C/H/O）
    opts.addAtomIndices = False
    opts.bondLineWidth = 2
    opts.highlightRadius = 0.42
    rdMolDraw2D.PrepareAndDrawMolecule(
        drawer,
        mol,
        highlightAtoms=highlight_atoms,
        highlightAtomColors=atom_cols,
        highlightBonds=highlight_bonds,
        highlightBondColors=bond_cols,
    )
    drawer.FinishDrawing()
    return drawer.GetDrawingText()


## Cliff 重現章節（指定 SMILES）

此章節用來重現你指定的 cliff 分子：

- `Cc1cc2c(s1)Nc1ccccc1N=C2N1CCN(C)CC1`

流程會：

1. 載入 `cliff` finetune checkpoint
2. 擷取 attention（softmax 後）
3. 輸出 0-1 固定色階的 atom heatmap
4. 輸出結構圖（紅色強度 + 0-1 colorbar）

In [ ]:
# ===== Cliff 參數 =====
CLIFF_SMILES = 'Cc1cc2c(s1)Nc1ccccc1N=C2N1CCN(C)CC1'
CLIFF_GPU_ID = '0'

# 請替換成你的 cliff 權重路徑
CLIFF_CKPT_PATH = './weights/cliff/CHEMBL231_Ki.pth'

# dist_bar 會從 checkpoint 還原，避免手動設定與模型訓練設定不一致。

# 分子 attention 顏色對比參數：gamma < 1 會放大高分區間，數值越小對比越強。
CLIFF_CONTRAST_GAMMA = 0.4

CLIFF_SAVE_PREFIX = 'cliff_demo'
SAVE_DIR = Path('./result/attention_vis')
SAVE_DIR.mkdir(parents=True, exist_ok=True)


def build_cliff_runtime_config(ckpt_path: str):
    cfg_path = Path(pdir) / 'config' / 'config_finetune_cliff.yaml'
    with open(cfg_path, 'r', encoding='utf-8') as f:
        cfg = yaml.load(f, Loader=yaml.FullLoader)
    cfg = config_current_user('cliff', cfg)
    cfg = config_dataset_form('pkl', cfg)
    cfg['checkpoint'] = ckpt_path
    return cfg


def restore_dist_bar_from_ckpt(ckpt_obj):
    state = ckpt_obj.get('model', ckpt_obj)
    if not isinstance(state, dict) or 'dist_bar' not in state:
        raise RuntimeError('Checkpoint does not contain dist_bar; cannot guarantee matching multiscale attention settings.')

    dist_bar = state['dist_bar']
    if torch.is_tensor(dist_bar):
        dist_bar = dist_bar.detach().cpu().numpy().tolist()
    elif hasattr(dist_bar, 'tolist'):
        dist_bar = dist_bar.tolist()

    GlobalVar.dist_bar = list(dist_bar)
    return GlobalVar.dist_bar


def build_cliff_model(cfg: Dict, device: str = 'cuda') -> torch.nn.Module:
    model = Scage(
        mode=cfg['mode'],  # cliff
        atom_names=CompoundKit.atom_vocab_dict.keys(),
        atom_embed_dim=cfg['model']['atom_embed_dim'],
        num_kernel=cfg['model']['num_kernel'],
        layer_num=cfg['model']['layer_num'],
        num_heads=cfg['model']['num_heads'],
        atom_FG_class=nfg() + 1,
        hidden_size=cfg['model']['hidden_size'],
        num_tasks=1,
    ).to(device)

    ckpt = torch.load(cfg['checkpoint'], map_location=device)
    state = ckpt.get('model', ckpt)
    state = {k[7:] if k.startswith('module.') else k: v for k, v in state.items()}
    cur = model.state_dict()
    cur.update({k: v for k, v in state.items() if k in cur})
    model.load_state_dict(cur)
    model.eval()
    return model


os.environ['CUDA_VISIBLE_DEVICES'] = CLIFF_GPU_ID
CLIFF_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

cliff_cfg = build_cliff_runtime_config(CLIFF_CKPT_PATH)
cliff_ckpt = torch.load(cliff_cfg['checkpoint'], map_location=CLIFF_DEVICE)
restored_dist_bar = restore_dist_bar_from_ckpt(cliff_ckpt)
print('restored dist_bar from checkpoint:', restored_dist_bar)
cliff_model = build_cliff_model(cliff_cfg, device=CLIFF_DEVICE)
cliff_batch = build_single_batch(CLIFF_SMILES, device=CLIFF_DEVICE)

cliff_attn_sink = []
cliff_hooks = register_attention_hooks(cliff_model, cliff_attn_sink)
with torch.no_grad():
    cliff_out = cliff_model(cliff_batch)
for h in cliff_hooks:
    h.remove()

cliff_graph_score = float(cliff_out['graph_feature'].view(-1).cpu().numpy()[0])
cliff_cls_logit = float(cliff_out['cliff_feature'].view(-1).cpu().numpy()[0])
cliff_cls_prob = float(torch.sigmoid(torch.tensor(cliff_cls_logit)).item())

cliff_n_atoms = int(cliff_batch['atom_length'][0].item())
cliff_raw_attn_figs = plot_raw_multiscale_attn_list(
    cliff_attn_sink,
    title_prefix=f'Raw MultiScaleAttention attn_list | {CLIFF_SMILES}',
)
(
    cliff_atom_atom_attn,
    cliff_atom_scores_raw,
    cliff_atom_scores,
    cliff_layer_mats,
) = aggregate_attention(cliff_attn_sink, n_atoms=cliff_n_atoms)

print('Cliff device:', CLIFF_DEVICE)
print('Cliff smiles:', CLIFF_SMILES)
print('graph_feature (reg):', cliff_graph_score)
print('cliff_feature logit:', cliff_cls_logit)
print('cliff_feature prob :', cliff_cls_prob)
print('num_atoms:', cliff_n_atoms)
print('atom_scores raw:', np.round(cliff_atom_scores_raw, 6))
print('atom_scores min-max:', np.round(cliff_atom_scores, 4))

Cliff device: cuda
Cliff smiles: Cc1cc2c(s1)Nc1ccccc1N=C2N1CCN(C)CC1
graph_feature (reg): -1.4466651678085327
cliff_feature logit: 0.2332114726305008
cliff_feature prob : 0.5580400824546814
num_atoms: 22
atom_scores (0-1): [0.0685 0.1506 0.0842 0.4892 0.3595 0.8286 0.5245 0.3153 0.0517 0.1311
 0.1311 0.0198 0.4392 0.9066 1.     0.6607 0.1351 0.0092 0.3838 0.2956
 0.     0.0199]


In [ ]:
# Cliff: 0-1 固定色階 heatmap（min-max normalization）
plt.figure(figsize=(6.2, 5.3))
sns.heatmap(
    cliff_atom_atom_attn,
    cmap='Blues',
    vmin=0.0,
    vmax=1.0,
    square=True,
    xticklabels=[str(i) for i in range(cliff_atom_atom_attn.shape[0])],
    yticklabels=[str(i) for i in range(cliff_atom_atom_attn.shape[0])],
)
plt.title('Cliff Atom Attention Heatmap (min-max)')
plt.xlabel('Atom index')
plt.ylabel('Atom index')
plt.tight_layout()
cliff_heatmap_path = SAVE_DIR / f'{CLIFF_SAVE_PREFIX}_heatmap.png'
plt.savefig(cliff_heatmap_path, dpi=220)
plt.show()
print('saved:', cliff_heatmap_path)
# Cliff: 結構圖 + 紅色 0-1 colorbar（min-max normalization）
cliff_img_bytes = draw_attention_on_molecule(
    CLIFF_SMILES,
    cliff_atom_scores,
    contrast_gamma=CLIFF_CONTRAST_GAMMA,
)
cliff_png_path = SAVE_DIR / f'{CLIFF_SAVE_PREFIX}_mol_attention.png'
with open(cliff_png_path, 'wb') as f:
    f.write(cliff_img_bytes)
fig, axes = plt.subplots(1, 2, figsize=(9.8, 4.8), gridspec_kw={'width_ratios': [20, 1]})
axes[0].imshow(plt.imread(str(cliff_png_path)))
axes[0].axis('off')
axes[0].set_title('Cliff Molecule Attention (min-max)')
norm = plt.Normalize(0.0, 1.0)
sm = plt.cm.ScalarMappable(cmap='Reds', norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, cax=axes[1])
cbar.set_label('Attention intensity (0-1)')
plt.tight_layout()
plt.show()
print('saved:', cliff_png_path)
print('\nAtom score list (raw vs min-max):')
for i, (s_raw, s_minmax) in enumerate(zip(cliff_atom_scores_raw, cliff_atom_scores)):
    print(f'atom {i:>2d}: raw={s_raw:.6f} | min-max={s_minmax:.4f}')


In [ ]:
# 自動迴圈：逐層生成 attention 圖（heatmap + 分子結構，僅顯示，不寫檔）
print('\nRender per-layer heatmaps and molecule attention plots (no file export) ...')
from io import BytesIO

n_layers = len(cliff_layer_mats)

# A) 每層 heatmap
heat_cols = min(3, n_layers)
heat_rows = int(np.ceil(n_layers / heat_cols))
fig, axes = plt.subplots(heat_rows, heat_cols, figsize=(5.2 * heat_cols, 4.6 * heat_rows))
axes = np.array(axes).reshape(-1)
for i in range(n_layers):
    sns.heatmap(
        cliff_layer_mats[i],
        cmap='Blues',
        vmin=0.0,
        vmax=1.0,
        square=True,
        cbar=(i == 0),
        ax=axes[i],
    )
    axes[i].set_title(f'Layer {i:02d} Heatmap (min-max)')
    axes[i].set_xlabel('Atom index')
    axes[i].set_ylabel('Atom index')
for j in range(n_layers, len(axes)):
    axes[j].axis('off')
plt.tight_layout()
plt.show()

# B) 每層分子結構 attention 圖（in-memory）
per_layer_imgs = []
for i, layer_mat in enumerate(cliff_layer_mats):
    layer_scores = layer_mat.mean(axis=0)
    layer_img_bytes = draw_attention_on_molecule(
        CLIFF_SMILES,
        layer_scores,
        contrast_gamma=CLIFF_CONTRAST_GAMMA,
    )
    per_layer_imgs.append(layer_img_bytes)
    print(f'layer {i:02d} rendered (in-memory)')
print(f'Total rendered: {len(per_layer_imgs)} layer plots')

show_cols = min(3, len(per_layer_imgs))
show_rows = int(np.ceil(len(per_layer_imgs) / show_cols))
fig, axes = plt.subplots(show_rows, show_cols, figsize=(6.2 * show_cols, 4.8 * show_rows))
axes = np.array(axes).reshape(-1)
for i, img_bytes in enumerate(per_layer_imgs):
    axes[i].imshow(plt.imread(BytesIO(img_bytes)))
    axes[i].axis('off')
    axes[i].set_title(f'Layer {i:02d} Molecule (min-max)')
for j in range(len(per_layer_imgs), len(axes)):
    axes[j].axis('off')
plt.tight_layout()
plt.show()
